[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/coopster-seclusion/sar-optical-pipeline/blob/main/notebooks/02_opera_retrieval.ipynb)

# Notebook 02: OPERA RTC-S1 Retrieval

Searches ASF DAAC for OPERA L2 RTC-S1 products matching the burst ID and orbit in `config.yaml`.
Downloads linear gamma0 GeoTIFFs, computes a temporal median composite per epoch (linear power domain),
converts to dB, clips to the study AOI, and writes outputs to Google Drive.

**Requires Colab Secrets:** `EARTHDATA_USERNAME`, `EARTHDATA_PASSWORD`

**Output:** `data/opera_rtc/opera_{year}_{pol}.tif` — Float32 dB rasters, clipped to AOI, nodata=NaN

**Before running:**
1. Set `opera.burst` in `config.yaml` — run Cell 2 (burst discovery) if you don't know your burst ID.
2. Set `opera.orbit` to `ascending` or `descending` (check [search.asf.alaska.edu](https://search.asf.alaska.edu)).
3. In Colab → Secrets panel, set `EARTHDATA_USERNAME` and `EARTHDATA_PASSWORD`.

In [ ]:
# ── Cell 1: run this first, every session ─────────────────────────────────────
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, subprocess, sys

REPO_URL = 'https://github.com/coopster-seclusion/sar-optical-pipeline.git'
REPO_DIR = '/content/sar-optical-pipeline'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

sys.path.insert(0, REPO_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'rioxarray', 'asf_search', 'hyp3_sdk', 'rasterstats', 'pyogrio'], check=True)

import numpy as np
import rioxarray as rxr
import geopandas as gpd
import matplotlib.pyplot as plt
import asf_search as asf

from pipeline.env import setup, repo_root
from pipeline.validate import validate_config
from pipeline.utils import resolve_crs
import pipeline.opera as opera_lib

cfg, DATA, OUT = setup(colab=True)
validate_config(cfg, repo_root())

SITE       = cfg['site_name']
DATA_OPERA = DATA / 'opera_rtc'
OUT_FIGS   = OUT  / 'figures'

print(f'Site        : {SITE}')
print(f'Data → OPERA: {DATA_OPERA}')
print(f'Burst ID    : {cfg["opera"]["burst"] or "NOT SET — run burst discovery below"}')
print(f'Orbit       : {cfg["opera"]["orbit"]}')
print(f'Epochs      : {[e["year"] for e in cfg["epochs"]]}')

## Burst discovery (run once for a new site)

If `opera.burst` is not yet set in `config.yaml`, run Cell 2 to find valid burst IDs for your AOI.
Copy the relevant ID into `config.yaml`, then re-run Cell 1.

Skip this cell if `config.yaml` already has a burst ID.

In [ ]:
# ── Cell 2: burst discovery ────────────────────────────────────────────────────
# Skip this cell if opera.burst is already set in config.yaml.
ROOT     = repo_root()
aoi_path = ROOT / cfg['aoi']['change_area']

burst_ids = opera_lib.suggest_burst_ids(
    aoi_path,
    orbit=cfg['opera']['orbit'],
)

## Authenticate and load AOI

In [ ]:
# ── Cell 3: authenticate Earthdata and load AOI ────────────────────────────────
username = userdata.get('EARTHDATA_USERNAME')
password = userdata.get('EARTHDATA_PASSWORD')

session = opera_lib.earthdata_session(username, password)
print('Earthdata session authenticated')

ROOT     = repo_root()
aoi_path = ROOT / cfg['aoi']['change_area']
aoi_gdf  = gpd.read_file(aoi_path).to_crs('EPSG:4326')
crs      = resolve_crs(cfg, aoi_path)

print(f'AOI file    : {aoi_path}')
print(f'AOI bounds  : {aoi_gdf.total_bounds.round(4)}  (W S E N)')
print(f'Output CRS  : {crs}')

## Search OPERA per epoch

Burst ID and orbit come from `config.yaml`. If any epoch returns 0 results, check:
- `opera.burst` matches the ASF catalog format exactly (underscores, e.g. `T010_020043_IW3`)
- `opera.orbit` is correct for your site
- The date window covers a period with S1 acquisitions

In [ ]:
# ── Cell 4: search OPERA per epoch ────────────────────────────────────────────
print(f'Burst: {cfg["opera"]["burst"]}  |  Orbit: {cfg["opera"]["orbit"].upper()}\n')

epoch_results = {}
for epoch in cfg['epochs']:
    year    = epoch['year']
    results = opera_lib.search_epoch(cfg, epoch['date_start'], epoch['date_end'])
    epoch_results[year] = results
    pol_info = results[0].properties.get('polarization', 'unknown') if results else 'n/a'
    print(f'  {year}: {len(results):>2} granule(s)   polarization={pol_info}')

if any(len(r) == 0 for r in epoch_results.values()):
    print('\nWARNING: one or more epochs returned 0 results.')
    print('Try running burst discovery (Cell 2) to verify the burst ID is correct.')

In [ ]:
# ── Cell 5: inspect result properties (diagnostic — run if download fails) ─────
# Shows what URL fields are present on a sample result so you can diagnose
# URL resolution issues. Safe to skip if processing completes successfully.
sample_year = cfg['epochs'][0]['year']
r_sample    = epoch_results.get(sample_year, [None])[0]
if r_sample:
    p = r_sample.properties
    print('url           :', p.get('url'))
    print('additionalUrls:', p.get('additionalUrls'))
    print('browse        :', p.get('browse'))
    print('polarization  :', p.get('polarization'))
    print('granuleName   :', p.get('granuleName'))
else:
    print(f'No results for {sample_year} — run Cell 4 first.')

## Download, composite, dB-convert, clip, save

Per epoch:
1. Downloads all granules for the date window to a temp directory
2. Temporal median composite in linear power domain (reduces speckle)
3. Converts to dB: `10 × log10(gamma0_linear)`; zero/negative → NaN
4. Reprojects to output CRS, clips to study AOI
5. Saves Float32 GeoTIFF; removes temp files

Skip logic: existing `opera_{year}_{pol}.tif` files are not re-downloaded.

In [ ]:
# ── Cell 6: process all epochs ────────────────────────────────────────────────
DATA_OPERA.mkdir(parents=True, exist_ok=True)

for epoch in cfg['epochs']:
    year = epoch['year']
    results = epoch_results[year]

    if not results:
        print(f'  {year}: 0 granules — skipping')
        continue

    print(f'  {year}: processing {len(results)} granule(s)...')
    outputs = opera_lib.process_epoch(
        year=year,
        results=results,
        aoi_path=aoi_path,
        crs=crs,
        out_dir=DATA,
        session=session,
        cfg=cfg,
    )
    for pol, path in outputs.items():
        arr = rxr.open_rasterio(path, masked=True).squeeze()
        vmin, vmax = np.nanpercentile(arr.values, [2, 98])
        print(f'    {pol}: shape={arr.shape}  2–98pct=[{vmin:.1f}, {vmax:.1f}] dB  → {path.name}')

print('\nDone.')

## Verification

Loads the baseline epoch and plots a greyscale preview for each polarization.
A clean image with visible land features confirms correct dB conversion and clipping.

In [ ]:
# ── Cell 7: verify baseline epoch ─────────────────────────────────────────────
baseline = cfg['change_detection']['baseline_year']
pols     = cfg['opera'].get('polarizations') or ['HH', 'HV']
n_pols   = len(pols)

fig, axes = plt.subplots(1, n_pols, figsize=(6 * n_pols, 5))
if n_pols == 1:
    axes = [axes]

any_found = False
for ax, pol in zip(axes, pols):
    path = DATA_OPERA / f'opera_{baseline}_{pol}.tif'
    if not path.exists():
        ax.set_title(f'{baseline} {pol} — NOT FOUND')
        ax.axis('off')
        print(f'Missing: {path}')
        continue

    arr  = rxr.open_rasterio(path, masked=True).squeeze()
    vmin, vmax = np.nanpercentile(arr.values, [2, 98])

    ax.imshow(arr.values, cmap='gray', vmin=vmin, vmax=vmax)
    ax.set_title(f'OPERA {baseline} {pol} (dB) — {SITE}')
    ax.axis('off')

    print(f'{baseline} {pol}:  shape={arr.shape}  CRS={arr.rio.crs}')
    print(f'  range=[{np.nanmin(arr.values):.1f}, {np.nanmax(arr.values):.1f}] dB  '
          f'2–98 pct=[{vmin:.1f}, {vmax:.1f}] dB')
    any_found = True

if any_found:
    plt.tight_layout()
    OUT_FIGS.mkdir(parents=True, exist_ok=True)
    save_path = OUT_FIGS / f'opera_{baseline}_preview.png'
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\nSaved: {save_path}')
else:
    print('No output files found — run processing cells above first.')